# Notebook 02 — Pipeline B: An Alternative Preprocessing Strategy

**Project:** Reproducible comparison of EEG preprocessing pipelines on OpenNeuro `ds004504`.

---

## What this notebook does

1. Presents the **literature review and gap analysis** that determined which alternative to test.
2. Implements **Pipeline B**, a deterministic, ICA-free artifact-handling strategy.
3. Runs the **identical** downstream analysis as Notebook 01 — same feature code, same classifier, same folds, same seed.
4. Measures signal quality and computational cost on the same axes.

## The one thing that must not go wrong

Under the fair-comparison rule, preprocessing is the **only** independent variable.
Everything downstream is imported from `ds004504_common.py`, the same module Notebook 01
used, so the two analyses cannot drift apart.

> **Prerequisite:** run `01_author_pipeline_ds004504.ipynb` first. This notebook reuses
> its configuration and subject selection.

## 1. Environment setup

### 1.1 Install packages

Identical to Notebook 01 — including `asrpy` and `mne-icalabel`, which Pipeline B does not use. Installing the same set keeps the environments identical so that runtime comparisons are not confounded by a different package stack.

In [ ]:
# Colab package installation.
# Versions are pinned loosely (>=) so the notebook keeps working as Colab's base image
# moves, but every resolved version is RECORDED later by get_environment_info().
%pip install -q "mne>=1.6" "scikit-learn>=1.3" "pandas>=2.0" "scipy>=1.10" \
                "matplotlib>=3.7" "pyarrow>=12.0" "psutil>=5.9" \
                "openneuro-py>=2024.1" "asrpy>=0.0.3" "mne-icalabel>=0.6" "onnxruntime>=1.16"

print("Installation finished. If Colab asks you to restart the runtime, do so and "
      "then re-run from this cell onwards.")

### 1.2 Mount Google Drive

In [ ]:
# Mounting Google Drive is what makes this experiment restartable: the cache and all
# result files live in Drive, so a Colab session timeout costs you nothing but the
# subject currently being processed.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as exc:
    IN_COLAB = False
    print(f"Not running in Colab ({exc.__class__.__name__}). "
          "Set OUTPUT_PATH below to a local directory instead.")

### 1.3 Load the shared analysis module

In [ ]:
# `ds004504_common.py` holds every pipeline, feature, cross-validation and statistics
# function. Both preprocessing notebooks import the SAME module, which is how the
# "fair comparison" requirement is enforced structurally rather than by convention.
import sys
from pathlib import Path

# Directory containing ds004504_common.py -- change if you put it elsewhere.
MODULE_DIR = "/content/drive/MyDrive/ds004504_experiment" if IN_COLAB else "."

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

module_file = Path(MODULE_DIR) / "ds004504_common.py"
if not module_file.exists():
    raise FileNotFoundError(
        f"ds004504_common.py not found at {module_file}.\n"
        "Upload it to that folder in Google Drive (or set MODULE_DIR to wherever you "
        "placed it) and re-run this cell."
    )

import ds004504_common as ds
print("Loaded ds004504_common version", ds.MODULE_VERSION)

### 1.4 Standard imports

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.facecolor": "white"})
print("Imports ready.")

### 1.5 Configuration

**Critical:** these values must match Notebook 01 exactly. Any difference in
`epoch_length_s`, `random_seed`, `cv_n_splits`, `resample_to`, `l_freq`, `h_freq` or
`reference` would break the fair comparison. The next cell verifies this against the
config Notebook 01 saved, rather than trusting you to remember.

In [ ]:
# =====================================================================================
# CONFIGURATION -- the single place to change how this notebook runs
# =====================================================================================
# RUN MODES (identical code path, only MAX_SUBJECTS changes):
#     Quick test  : MAX_SUBJECTS = 5     (smoke test; too few for real statistics)
#     Development : MAX_SUBJECTS = 20
#     Full        : MAX_SUBJECTS = None  (all 88 subjects)
# Must match Notebook 01. The next cell verifies this automatically.

DATASET_PATH = "/content/ds004504"
OUTPUT_PATH  = "/content/drive/MyDrive/ds004504_experiment" if IN_COLAB else "./ds004504_experiment"
MAX_SUBJECTS = 20            # <-- change this to switch run mode
RANDOM_SEED  = 42
N_JOBS       = 1             # Colab free tier has ~2 vCPUs; 1 keeps peak RAM predictable

cfg = ds.Config(
    dataset_path=DATASET_PATH,
    output_path=OUTPUT_PATH,
    max_subjects=MAX_SUBJECTS,
    random_seed=RANDOM_SEED,
    n_jobs=N_JOBS,
    # --- preprocessing parameters shared by BOTH pipelines (controlled variables) ---
    l_freq=0.5, h_freq=45.0, butter_order=4,
    resample_to=100.0,       # 500 -> 100 Hz; safe below the 45 Hz low-pass, 5x cheaper
    reference="average",     # see ds.NOTE_ON_REFERENCE for the A1-A2 caveat
    # --- epoching / features (identical for both pipelines) ---
    epoch_length_s=4.0, epoch_overlap_s=0.0,
    # --- cross-validation ---
    cv_n_splits=10, cv_n_repeats=5, run_loso=True,
    n_bootstrap=10000,
    cache_enabled=True, overwrite_cache=False,
)
cfg.make_dirs()

print(f"Run mode      : {cfg.mode()}")
print(f"Max subjects  : {cfg.max_subjects}")
print(f"Output path   : {cfg.out}")
print(f"Cache path    : {cfg.cache_dir}")
saved = cfg.save()
print(f"Config saved  : {saved}")

### 1.6 Verify configuration matches Notebook 01

An automated guard: if any controlled variable differs from the run that produced Pipeline A's results, this cell fails loudly instead of silently producing an invalid comparison.

In [ ]:
CONTROLLED = ["l_freq", "h_freq", "butter_order", "resample_to", "reference",
              "epoch_length_s", "epoch_overlap_s", "drop_first_s", "psd_n_fft_s",
              "random_seed", "cv_n_splits", "cv_n_repeats", "epoch_reject_z",
              "primary_classifier", "secondary_classifier"]

prev_path = cfg.results_dir / "config.json"
if not prev_path.exists():
    raise FileNotFoundError(
        f"No config.json found at {prev_path}. Run Notebook 01 first -- Pipeline B's "
        "results are only interpretable relative to Pipeline A's."
    )

with open(prev_path) as fh:
    prev = json.load(fh)

mismatches = {}
for key in CONTROLLED:
    now, before = getattr(cfg, key), prev.get(key)
    if now != before:
        mismatches[key] = {"notebook_01": before, "notebook_02": now}

if mismatches:
    print("CONFIGURATION MISMATCH -- the comparison would NOT be fair:\n")
    for k, v in mismatches.items():
        print(f"  {k}: Notebook 01 = {v['notebook_01']!r}, here = {v['notebook_02']!r}")
    raise ValueError(
        "Controlled variables differ from Notebook 01. Align them, or re-run "
        "Notebook 01 with the new settings, before continuing."
    )

print("PASS - all controlled variables match Notebook 01.")
print(f"  Filter      : {cfg.l_freq}-{cfg.h_freq} Hz, Butterworth order {cfg.butter_order}")
print(f"  Reference   : {cfg.reference}")
print(f"  Resample    : {cfg.resample_to} Hz")
print(f"  Epochs      : {cfg.epoch_length_s} s, overlap {cfg.epoch_overlap_s} s")
print(f"  Seed        : {cfg.random_seed}")
print(f"  CV          : {cfg.cv_n_splits}-fold x {cfg.cv_n_repeats} repeats")

### 1.7 Environment and logging

In [ ]:
# Capture the exact execution environment. Runtime comparisons are only meaningful
# within one environment, so this record is what lets a reader judge our timings.
import json
env = ds.get_environment_info()
ds.save_results(cfg, "environment", env)

print(f"Python      : {env['python_version']}")
print(f"CPU         : {env.get('cpu_model', 'unknown')}")
print(f"Logical CPUs: {env['cpu_count_logical']}")
print(f"RAM total   : {env['ram_total_gb']} GB")
print(f"GPU         : {env['gpu']}  (not used -- this experiment is CPU-only by design)")
print("\nKey package versions:")
for pkg in ["mne", "numpy", "scipy", "scikit-learn", "sklearn", "asrpy", "mne_icalabel"]:
    if pkg in env["packages"]:
        print(f"  {pkg:<15} {env['packages'][pkg]}")

In [ ]:
log_path = ds.setup_logging(cfg, name="pipeline_B")
print("Logging to:", log_path)

---
## 2. Literature review and gap analysis

This section is the justification for Pipeline B. It was completed **before** any
alternative was implemented.

### 2.1 How much prior work exists on this dataset?

`ds004504` has become the most widely used open EEG resource for dementia classification.
A 2026 scoping review — published by the dataset's own authors — systematically catalogued
every machine-learning study applying it:

> Miltiadous, A., Ntetska, A., Aspiotis, V., Moustakli, E., Tsipouras, M. G., Tzallas,
> A. T., Giannakeas, N., Glavas, E., Angelidis, P., & Tzimourta, K. D. (2026).
> *The AHEPA EEG benchmark: setting the standard for machine learning in dementia
> diagnosis, a scoping review.* **Cognitive Neurodynamics**, 20(1), 95.
> https://doi.org/10.1007/s11571-026-10464-w

Its findings that matter for us:

- **46 studies** were included, screened from 112 Scopus citations.
- Studies were graded into three validity tiers. Only **12 of 46** reached Validity 1 (subject-level validation with no leakage). **22 of 46** fell into Validity 3 (leakage present).
- Methodological rigour is **inversely** related to reported accuracy: for AD vs CN, the mean accuracy falls from 90.8% across all studies to **82.1%** among Validity-1 studies. Each step down in validation rigour is associated with roughly **7–10 percentage points** of accuracy inflation, explaining over half the variance in reported performance.
- Under proper validation, **traditional classifiers matched deep models**, suggesting much of deep learning's apparent advantage on this dataset comes from leakage rather than representational power.
- Their criterion **C4** is *"Transparency of EEG preprocessing"* — an explicit acknowledgement that preprocessing reporting is a weak point across this literature.

### 2.2 The gap

Two observations define the gap this project occupies.

**First**, the review states in its own limitations that it *"did not perform quantitative
re-estimation of reported performance metrics, as no re-analysis of raw datasets or
simulation of alternative validation schemes was conducted"*. The benchmark is a synthesis
of *published numbers*, not of re-executed pipelines.

**Second**, and more decisively: essentially all 46 studies consume the authors'
**pre-cleaned derivative files**. Preprocessing is treated as a fixed, inherited
constant — never as an experimental variable. No study in the catalogue re-runs
preprocessing from the raw recordings and asks what the choice of pipeline is worth.

So: preprocessing on this dataset is simultaneously (a) universally inherited, (b)
under-reported by the review's own criterion, and (c) never empirically evaluated.

### 2.3 Candidate alternatives and whether each is already taken

| Alternative | Used on ds004504 before? | Paper | What was changed? | Reported outcome | Novel comparison still possible? |
|---|---|---|---|---|---|
| ASR + RunICA + ICLabel (0.5–45 Hz, A1–A2) | Yes — this **is** the authors' pipeline | Miltiadous et al. 2023, *Data* 8(6):95 | n/a — the reference pipeline | Supplied as derivatives; inherited by ~all 46 studies | This is our **Pipeline A**, not an alternative |
| Minimal preprocessing (filter only) | Not identified on ds004504 | Delorme 2023, *Sci Rep* 13:2372 (other datasets) | Removed all automated correction | Filtering + bad-channel interpolation performed as well as or better than heavier pipelines | **Yes** — untested here, and never for a diagnostic endpoint |
| Bad-channel detection + interpolation | Not identified on ds004504 | Bigdely-Shamlo et al. 2015 (PREP) | Robust referencing + interpolation | One of only two steps Delorme found beneficial | **Yes** |
| ASR alone (no ICA) | Not identified on ds004504 | Chang et al. 2020; Callan et al. 2024 | Ablate the ICA stage | ASR effective for transients; ICA adds most value under extreme artifact | Partly — but overlaps Pipeline A |
| ICA + MARA instead of ICLabel | Not identified on ds004504 | Winkler et al. 2011 | Swap the IC classifier | Comparable classification | Yes, but keeps the expensive ICA stage, so it cannot address the cost question |
| Wavelet thresholding / wavelet-ICA | Not identified on ds004504 | Mammone et al. 2012 | Wavelet denoising of ICs | Good morphology preservation | Yes, but adds parameters and cost |
| Riemannian / robust PCA denoising | Not identified on ds004504 | Blum et al. 2019 | Riemannian ASR | Improved robustness | Yes, but no mature Python implementation for Colab |

### 2.4 Which alternative, and why

**Selected: a deterministic, ICA-free artifact-handling stage.**

```
Raw EEG
  → Butterworth band-pass 0.5–45 Hz        [IDENTICAL to Pipeline A]
  → Re-reference                            [IDENTICAL to Pipeline A]
  → Trim onset + resample to 100 Hz         [IDENTICAL to Pipeline A]
  → Deterministic bad-channel detection     ← the only difference
  → Spherical-spline interpolation          ← the only difference
  → Robust epoch rejection                  [IDENTICAL to Pipeline A]
  → cleaned EEG
```

The reasoning:

1. **It isolates one variable.** Filtering, referencing, resampling, epoching, features, classifier and folds are all held constant. The *only* thing that differs is the artifact-removal stage. A pipeline that also changed the filter would confound the two.

2. **It targets the actual cost centre.** Notebook 01 §5.1 shows where Pipeline A's runtime goes: ASR and Infomax ICA dominate. Replacing exactly that stage is what makes the computational question answerable.

3. **It has direct literature support.** Delorme (2023) found across three public collections that, apart from high-pass filtering and bad-channel interpolation, automated corrections — including automated ICA rejection of eye and muscle components — did not reliably improve data quality. Pipeline B is essentially "the two steps that survived that analysis, and nothing else."

4. **The question is genuinely open.** de Cheveigné (2023) rebuts Delorme's metric as too narrow, arguing that the value of a cleaning tool depends on whether its target artifact is present. That disagreement is precisely why this needs testing empirically rather than being asserted. Note also that Delorme's endpoint was ERP detectability in task data; ours is diagnostic classification from resting-state data. **We are not assuming his conclusion transfers — we are testing whether it does.**

5. **It is fully deterministic.** Infomax ICA depends on random initialisation, so Pipeline A gives a different answer on re-run unless the seed is fixed — and the authors' seed is unavailable. Pipeline B has no stochastic component at all. This directly addresses the project's robustness/reproducibility objective.

### 2.5 What we are and are not claiming

Pipeline B is **not novel as a method**. Bad-channel interpolation and amplitude-based
epoch rejection are standard, decades-old techniques. Nothing here is invented.

The contribution is the **comparison**, described conservatively:

- an *empirical comparison* of two preprocessing strategies on ds004504
- a *reproducibility study* of the authors' published pipeline
- a *computational benchmark* under fixed, resource-constrained conditions

We do **not** claim "first ever" or "novel method". We claim this specific controlled
comparison does not appear in the 46 catalogued studies, which is a statement about the
literature, not about the sophistication of the technique.

### 2.6 Falsifiable hypotheses

| # | Hypothesis | How it could be falsified |
|---|---|---|
| H1 | Pipeline B is substantially cheaper per subject | Measured runtimes are comparable |
| H2 | Classification performance is *comparable* (CI for the difference includes 0) | The CI excludes 0 in either direction |
| H3 | Pipeline B preserves spectral content (relative band powers similar) | Relative band powers differ markedly |
| H4 | Pipeline B leaves more residual high-amplitude activity | Residual artifact indicators are equal or lower |

H2 is deliberately a hypothesis of **equivalence, not superiority**. We are not trying to
make Pipeline B win.

In [ ]:
# Persist the gap analysis as a machine-readable table (brief section 4).
novelty_rows = [
    {"alternative": "ASR + RunICA + ICLabel (authors' pipeline)",
     "used_on_ds004504": "Yes - it IS the reference pipeline",
     "paper": "Miltiadous et al. 2023, Data 8(6):95",
     "what_changed": "n/a (reference)",
     "reported_outcome": "Supplied as derivatives; inherited by ~all 46 catalogued studies",
     "novel_comparison_possible": "This is Pipeline A, not an alternative"},
    {"alternative": "Minimal preprocessing (filter only)",
     "used_on_ds004504": "Not identified",
     "paper": "Delorme 2023, Sci Rep 13:2372 (other datasets)",
     "what_changed": "Removed all automated correction",
     "reported_outcome": "Filtering + bad-channel interpolation as good as or better than heavier pipelines",
     "novel_comparison_possible": "Yes"},
    {"alternative": "Bad-channel detection + interpolation",
     "used_on_ds004504": "Not identified",
     "paper": "Bigdely-Shamlo et al. 2015 (PREP)",
     "what_changed": "Robust referencing + interpolation",
     "reported_outcome": "One of two steps Delorme found beneficial",
     "novel_comparison_possible": "Yes - SELECTED"},
    {"alternative": "ASR alone (no ICA)",
     "used_on_ds004504": "Not identified",
     "paper": "Chang et al. 2020; Callan et al. 2024",
     "what_changed": "Ablate ICA stage",
     "reported_outcome": "ASR effective for transients",
     "novel_comparison_possible": "Partly - overlaps Pipeline A"},
    {"alternative": "ICA + MARA instead of ICLabel",
     "used_on_ds004504": "Not identified",
     "paper": "Winkler et al. 2011",
     "what_changed": "Swap IC classifier",
     "reported_outcome": "Comparable classification",
     "novel_comparison_possible": "Yes, but retains expensive ICA - cannot address cost"},
    {"alternative": "Wavelet thresholding / wavelet-ICA",
     "used_on_ds004504": "Not identified",
     "paper": "Mammone et al. 2012",
     "what_changed": "Wavelet denoising of ICs",
     "reported_outcome": "Good morphology preservation",
     "novel_comparison_possible": "Yes, but adds parameters and cost"},
    {"alternative": "Riemannian ASR / robust PCA",
     "used_on_ds004504": "Not identified",
     "paper": "Blum et al. 2019",
     "what_changed": "Riemannian ASR variant",
     "reported_outcome": "Improved robustness",
     "novel_comparison_possible": "Yes, but no mature Colab-ready Python implementation"},
]
novelty = pd.DataFrame(novelty_rows)
ds.save_results(cfg, "novelty_assessment", novelty)

hypotheses = pd.DataFrame([
    {"id": "H1", "hypothesis": "Pipeline B is substantially cheaper per subject",
     "falsified_if": "Measured runtimes are comparable"},
    {"id": "H2", "hypothesis": "Classification performance is comparable (CI includes 0)",
     "falsified_if": "CI for the difference excludes 0 in either direction"},
    {"id": "H3", "hypothesis": "Pipeline B preserves spectral content",
     "falsified_if": "Relative band powers differ markedly"},
    {"id": "H4", "hypothesis": "Pipeline B leaves more residual high-amplitude activity",
     "falsified_if": "Residual artifact indicators are equal or lower"},
])
ds.save_results(cfg, "hypotheses", hypotheses)

print("SEARCH SCOPE: 46 ML studies catalogued by the AHEPA benchmark review "
      "(Scopus citations of the data descriptor, screened to 2025-08-26), plus targeted "
      "searches for preprocessing-comparison work on this dataset.")
print("\nCONCLUSION: no catalogued study treats preprocessing as the independent "
      "variable or re-runs it from the raw recordings.\n")
display(novelty[["alternative", "used_on_ds004504", "novel_comparison_possible"]])
print()
display(hypotheses)

---
## 3. Dataset and subject selection

We reuse the same dataset and the **same subject list** as Notebook 01. Using a different
subset would break the pairing that the statistical comparison depends on.

In [ ]:
# We download ONLY the subjects this run needs, plus the small top-level metadata.
# openneuro-py skips files that already exist, so re-running after a session restart
# is cheap and safe.
import openneuro
from pathlib import Path

Path(cfg.dataset_path).mkdir(parents=True, exist_ok=True)

# Step 1: top-level metadata only (a few kB) -- needed to know who the subjects are.
try:
    openneuro.download(dataset="ds004504", target_dir=cfg.dataset_path,
                       include=["participants.tsv", "participants.json",
                                "dataset_description.json", "README"])
    print("Metadata downloaded.")
except Exception as exc:
    print(f"Metadata download failed: {exc!r}")
    print("If this persists, download ds004504 manually and point DATASET_PATH at it.")

In [ ]:
# Subject IDs in this dataset are ordered by diagnosis, so naively taking "the first N"
# would return an all-Alzheimer's subset with no controls and make classification
# impossible. We therefore select a GROUP-STRATIFIED subset for reduced-size runs.
subjects = ds.select_subjects_balanced(cfg)

participants = pd.read_csv(Path(cfg.dataset_path) / "participants.tsv", sep="\t")
group_col = ds._find_group_column(participants)
groups = dict(zip(participants["participant_id"].astype(str),
                  participants[group_col].astype(str)))

sel_groups = pd.Series([groups.get(s, "?") for s in subjects]).value_counts().to_dict()
print(f"Selected {len(subjects)} subjects: {sel_groups}")
print(f"Group label meaning: {ds.GROUP_LABELS}")
print("First few:", subjects[:8])

### 3.1 Confirm the subject list matches Notebook 01

The paired statistics in Notebook 03 require both pipelines to have scored the *same people*.

In [ ]:
status_A_path = cfg.results_dir / "processing_status_A.csv"
if status_A_path.exists():
    status_A_prev = pd.read_csv(status_A_path)
    subjects_A = set(status_A_prev["participant_id"])
    subjects_B = set(subjects)
    only_A, only_B = subjects_A - subjects_B, subjects_B - subjects_A
    print(f"Notebook 01 processed : {len(subjects_A)} subjects")
    print(f"This notebook selects : {len(subjects_B)} subjects")
    print(f"Overlap               : {len(subjects_A & subjects_B)}")
    if only_A or only_B:
        print(f"\nWARNING - subject lists differ.")
        if only_A: print(f"  Only in Notebook 01: {sorted(only_A)}")
        if only_B: print(f"  Only here          : {sorted(only_B)}")
        print("  Notebook 03 will restrict the paired comparison to the intersection.")
    else:
        print("\nPASS - identical subject lists.")
else:
    print("Notebook 01 status file not found. Run Notebook 01 first.")

---
## 4. Pipeline B on a single subject

### The three components

**1. Bad-channel detection (deterministic).** Two criteria, both computed with
median/MAD statistics so that the criteria are not themselves driven by the outliers they
are meant to find:

- *Low correlation* — a channel whose maximum absolute correlation with any other channel falls below 0.4 is not sharing the volume-conducted signal every scalp electrode should see, indicating a disconnected or bridged electrode.
- *Amplitude deviation* — a channel whose robust z-scored amplitude exceeds 5 is flat or noise-dominated.

A safety valve caps the number of channels that may be flagged at 25% of the montage; if
more than that look bad, the criteria are more likely wrong than the data, so the subject
is flagged rather than having most of its montage synthesised.

**2. Spherical-spline interpolation.** Bad channels are reconstructed from their
neighbours using the electrode geometry. With only 19 widely spaced electrodes,
interpolation is less accurate than in high-density montages — a limitation we record
explicitly.

**3. Robust epoch rejection** (applied at the epoching stage, to *both* pipelines).
The threshold is derived per subject from the median and MAD of the epoch peak-to-peak
distribution. A fixed microvolt threshold would not transfer across subjects with
different impedances and different disease-related amplitudes.

In [ ]:
demo_sub = subjects[0]
print(f"Demonstration subject: {demo_sub} (group {groups[demo_sub]})")

raw_demo = ds.load_raw_subject(cfg, demo_sub, preload=True)
raw_before = raw_demo.copy()

clean_B, meta_B = ds.pipeline_b(raw_demo, cfg)

bi = meta_B["bad_channel_info"]
print(f"\nBad channels detected : {bi['n_bad_channels']} -> {bi['bad_channels']}")
print(f"  by correlation      : {bi['bad_by_correlation']}")
print(f"  by amplitude        : {bi['bad_by_deviation']}")
print(f"  cap exceeded        : {bi['excessive_bad_channels']}")
print(f"Interpolation applied : {meta_B['interpolated']}")
print(f"Total wall time       : {meta_B['total_wall_time_s']:.2f} s")
if meta_B["warnings"]:
    print("\nWARNINGS:")
    for w in meta_B["warnings"]:
        print("  -", w)

### 4.1 Cost breakdown, and the direct comparison against Pipeline A

In [ ]:
steps_B = pd.DataFrame(meta_B["steps"]).T[["wall_time_s", "cpu_time_s", "peak_rss_mb"]]
steps_B["pct_of_total"] = (steps_B["wall_time_s"] / steps_B["wall_time_s"].sum() * 100).round(1)
display(steps_B)

# Same subject, Pipeline A, from the cache Notebook 01 wrote.
meta_A_path = cfg.cache_dir / "pipeline_A" / f"{demo_sub}_meta.json"
if meta_A_path.exists():
    with open(meta_A_path) as fh:
        mA = json.load(fh)
    tA = mA.get("preprocess_meta", {}).get("total_wall_time_s")
    tB = meta_B["total_wall_time_s"]
    if tA:
        print(f"\nSame subject, preprocessing wall time:")
        print(f"  Pipeline A : {tA:6.2f} s")
        print(f"  Pipeline B : {tB:6.2f} s")
        print(f"  Speed-up   : {tA / tB:5.1f}x  (single subject -- the batch figure "
              f"in section 8 is the one to quote)")
else:
    print("\nPipeline A cache not found for this subject; run Notebook 01 first.")

### 4.2 Raw versus cleaned EEG

In [ ]:
def plot_before_after(raw_b, raw_a, title_b, title_a, seconds=10, n_ch=8, fname=None):
    """Same time window before and after cleaning, shared amplitude scale."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    for ax, r, ttl in zip(axes, [raw_b, raw_a], [title_b, title_a]):
        sf = r.info["sfreq"]
        n = int(seconds * sf)
        start = min(int(30 * sf), max(0, r.n_times - n))
        data = r.get_data(start=start, stop=start + n)[:n_ch] * 1e6
        t = np.arange(data.shape[1]) / sf
        offset = 120
        for i in range(data.shape[0]):
            ax.plot(t, data[i] + i * offset, lw=0.6, color="#2b2b2b")
        ax.set_yticks([i * offset for i in range(data.shape[0])])
        ax.set_yticklabels(r.ch_names[:n_ch])
        ax.set_xlabel("Time (s)")
        ax.set_title(ttl)
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Channel (traces offset by 120 µV for legibility)")
    plt.tight_layout()
    if fname:
        plt.savefig(cfg.figures_dir / fname, dpi=150, bbox_inches="tight")
    plt.show()

plot_before_after(raw_before, clean_B,
                  f"Raw EEG — {demo_sub}", f"After Pipeline B — {demo_sub}",
                  fname="example_raw_vs_pipelineB.png")
print("Interpretation: Pipeline B is expected to preserve more of the original waveform "
      "than Pipeline A, because it removes no components and reconstructs no subspaces. "
      "Whether that is a virtue (signal preserved) or a fault (artifact retained) is the "
      "empirical question this project answers.")

### 4.3 Three-way spectral comparison: raw, Pipeline A, Pipeline B

This single figure is the most informative diagnostic in the whole project — it shows what each pipeline does to the spectrum of the same recording.

In [ ]:
import scipy.signal as sps

def mean_psd(raw):
    sf = raw.info["sfreq"]
    nfft = int(min(4 * sf, raw.n_times))
    f, p = sps.welch(raw.get_data(), fs=sf, nperseg=nfft, noverlap=nfft // 2, axis=-1)
    return f, p.mean(axis=0)

series = [(raw_before, "Raw (unprocessed)", ":", "grey"),
          (clean_B, "Pipeline B (alternative)", "-", "#dd8452")]

cache_A = cfg.cache_dir / "pipeline_A" / f"{demo_sub}_meta.json"
if cache_A.exists():
    # Re-run Pipeline A on this one subject so both spectra come from the same session.
    try:
        clean_A_demo, _ = ds.pipeline_a(ds.load_raw_subject(cfg, demo_sub), cfg)
        series.insert(1, (clean_A_demo, "Pipeline A (authors')", "-", "#4c72b0"))
    except Exception as exc:
        print(f"Could not recompute Pipeline A for the figure: {exc!r}")

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for raw_obj, lab, style, colour in series:
    f, p = mean_psd(raw_obj)
    m = (f >= 0.5) & (f <= 45)
    ax.semilogy(f[m], p[m], style, lw=1.8, label=lab, color=colour)

for name, (lo, hi) in ds.FREQ_BANDS.items():
    ax.axvline(lo, color="grey", lw=0.5, alpha=0.4)
    ax.text(lo + 0.2, ax.get_ylim()[1] * 0.5, name, fontsize=7, rotation=90,
            color="grey", va="top")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power spectral density (V²/Hz)")
ax.set_title(f"PSD by pipeline — subject {demo_sub} (channel-averaged)")
ax.legend()
plt.tight_layout()
plt.savefig(cfg.figures_dir / "psd_pipeline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("What to look for: if the two cleaned spectra overlap closely, the pipelines are "
      "extracting equivalent spectral information and any classification difference must "
      "come from elsewhere. Divergence localised to a band points to where the pipelines "
      "genuinely disagree.")

---
## 5. Batch processing

Same driver, same caching, same error handling as Notebook 01 — only the pipeline
argument changes.

In [ ]:
for name in ["raw_before", "clean_B", "raw_demo", "clean_A_demo"]:
    if name in dir():
        del globals()[name]
import gc; gc.collect()

status_B = ds.process_many_subjects(cfg, subjects, pipeline="B", groups=groups)
ds.save_results(cfg, "processing_status_B", status_B)

n_ok = int(status_B["status"].isin(["ok", "cached"]).sum())
print(f"\nSucceeded: {n_ok}/{len(subjects)}")

### 5.1 Exclusion report

In [ ]:
ok_mask = status_B["status"].isin(["ok", "cached"])
failed = status_B.loc[~ok_mask, ["participant_id", "group", "status", "error"]]

report_B = {
    "total_subjects_expected": len(subjects),
    "successfully_processed": int(ok_mask.sum()),
    "failed": int((~ok_mask).sum()),
    "failed_subject_ids": failed["participant_id"].tolist(),
    "failure_reasons": dict(zip(failed["participant_id"], failed["error"])),
}
ds.save_results(cfg, "exclusion_report_B", report_B)

print(f"Expected              : {report_B['total_subjects_expected']}")
print(f"Successfully processed: {report_B['successfully_processed']}")
print(f"Failed                : {report_B['failed']}")
if report_B["failed"]:
    for sid, err in report_B["failure_reasons"].items():
        print(f"  {sid}: {err}")
else:
    print("\nNo failures.")

subjects_ok_B = status_B.loc[ok_mask, "participant_id"].tolist()

### 5.2 Bad-channel statistics

How often did Pipeline B actually intervene? If it almost never flags a channel, it is effectively a filter-only pipeline, and the comparison should be described that way.

In [ ]:
rows = []
for sid in subjects_ok_B:
    mp = cfg.cache_dir / "pipeline_B" / f"{sid}_meta.json"
    if not mp.exists():
        continue
    with open(mp) as fh:
        m = json.load(fh)
    bi = m.get("preprocess_meta", {}).get("bad_channel_info", {})
    rows.append({
        "participant_id": sid,
        "group": m.get("group"),
        "n_bad_channels": bi.get("n_bad_channels", 0),
        "bad_channels": "|".join(bi.get("bad_channels", [])),
        "excessive": bi.get("excessive_bad_channels", False),
        "n_epochs": m.get("n_epochs"),
        "epochs_dropped": m.get("epoch_info", {}).get("epoch_rejection", {}).get("n_dropped"),
    })

badch = pd.DataFrame(rows)
ds.save_results(cfg, "bad_channel_stats_B", badch)

if not badch.empty:
    print(f"Subjects with >=1 bad channel : {(badch['n_bad_channels'] > 0).sum()}/{len(badch)}")
    print(f"Mean bad channels per subject : {badch['n_bad_channels'].mean():.2f} of 19")
    print(f"Max bad channels              : {badch['n_bad_channels'].max()}")
    print(f"Subjects hitting the 25% cap  : {badch['excessive'].sum()}")
    print(f"\nBy group:")
    display(badch.groupby("group")["n_bad_channels"].agg(["mean", "max", "count"]).round(2))

    all_bad = [c for s in badch["bad_channels"] if s for c in s.split("|")]
    if all_bad:
        print("\nMost frequently interpolated channels:")
        display(pd.Series(all_bad).value_counts().head(8))
    else:
        print("\nNo channels were flagged in any subject. Pipeline B therefore reduces "
              "to filter + epoch rejection for this cohort -- an important finding to "
              "report honestly rather than a failure.")
else:
    print("No Pipeline B metadata available.")

### 5.3 Signal-quality results

In [ ]:
rows = []
for sid in subjects_ok_B:
    mp = cfg.cache_dir / "pipeline_B" / f"{sid}_meta.json"
    if not mp.exists():
        continue
    with open(mp) as fh:
        m = json.load(fh)
    before, after = m.get("quality_before", {}), m.get("quality_after", {})
    if not after:
        continue
    row = {"participant_id": sid, "group": m.get("group"), "pipeline": "B"}
    for k, v in after.items():
        if isinstance(v, (int, float)):
            row[f"after_{k}"] = v
            if isinstance(before.get(k), (int, float)):
                row[f"before_{k}"] = before[k]
    rows.append(row)

quality_B = pd.DataFrame(rows)
ds.save_results(cfg, "signal_quality_results_B", quality_B)

show = [c for c in ["participant_id", "group", "before_rms_v", "after_rms_v",
                    "before_frac_samples_robust_z_gt5", "after_frac_samples_robust_z_gt5",
                    "after_relpow_alpha"] if c in quality_B.columns]
display(quality_B[show].head(12))
print(f"\nSignal-quality table: {quality_B.shape[0]} subjects x {quality_B.shape[1]} columns")

---
## 6. Downstream analysis — identical to Notebook 01

Every call below uses the same shared functions, the same feature list, the same
classifier, the same folds and the same seed as Notebook 01. Only the input features
differ, because only the preprocessing differed.

In [ ]:
features_B = ds.assemble_feature_table(cfg, "B", subjects_ok_B)
feature_names_B = [c for c in features_B.columns
                   if c not in ("participant_id", "group", "epoch_index")]

# Guard: the feature space must be identical, or the classifiers are not comparable.
fn_path = cfg.results_dir / "feature_names.json"
if fn_path.exists():
    with open(fn_path) as fh:
        feature_names_A = json.load(fh)["feature_names"]
    if feature_names_A != feature_names_B:
        only_A = set(feature_names_A) - set(feature_names_B)
        only_B = set(feature_names_B) - set(feature_names_A)
        raise ValueError(
            f"Feature spaces differ between pipelines!\n"
            f"  Only in A: {sorted(only_A)[:10]}\n"
            f"  Only in B: {sorted(only_B)[:10]}\n"
            "The comparison would not be fair. Investigate before continuing."
        )
    print(f"PASS - feature space identical to Pipeline A ({len(feature_names_B)} features)")
else:
    print("Notebook 01 feature list not found; cannot verify. Run Notebook 01 first.")

feature_names = feature_names_B
print(f"\nFeature matrix : {features_B.shape[0]} epochs x {len(feature_names)} features")
print(f"Subjects       : {features_B['participant_id'].nunique()}")
print(f"\nSubjects per group:\n{features_B.groupby('group')['participant_id'].nunique()}")

### 6.1 Epoch-count comparison

If the two pipelines retain very different numbers of epochs, that is itself a result — it means one pipeline is discarding more data, which affects both statistical power and what "comparable performance" means.

In [ ]:
counts_B = features_B.groupby("participant_id").size().rename("epochs_B")
try:
    fa_path_ok = True
    features_A_chk = ds.assemble_feature_table(cfg, "A", subjects_ok_B)
    counts_A = features_A_chk.groupby("participant_id").size().rename("epochs_A")
    cmp_counts = pd.concat([counts_A, counts_B], axis=1).dropna()
    cmp_counts["difference_B_minus_A"] = cmp_counts["epochs_B"] - cmp_counts["epochs_A"]
    display(cmp_counts.describe().round(1))
    print(f"\nTotal epochs  A: {int(cmp_counts['epochs_A'].sum())}  "
          f"B: {int(cmp_counts['epochs_B'].sum())}")
    print(f"Mean difference (B - A): {cmp_counts['difference_B_minus_A'].mean():.1f} epochs/subject")
    print("\nA large positive difference means Pipeline A discarded more data (ASR "
          "shortens recordings). This is expected and must be acknowledged when "
          "interpreting any performance difference.")
    del features_A_chk
except Exception as exc:
    print(f"Could not load Pipeline A features for comparison: {exc!r}")

### 6.2 Leakage verification

Re-run on Pipeline B's own data. The guarantee has to hold here too, not just in Notebook 01.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sub = features_B[features_B["group"].isin(["A", "C"])]
gg, yy = sub["participant_id"].to_numpy(), sub["group"].to_numpy()
n_per_class = sub.groupby("group")["participant_id"].nunique()
n_splits = min(cfg.cv_n_splits, int(n_per_class.min()))

violations = 0
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed)
for tr, te in sgkf.split(np.zeros(len(yy)), yy, gg):
    if set(gg[tr]) & set(gg[te]):
        violations += 1

print(f"Subjects per class      : {n_per_class.to_dict()}")
print(f"Folds checked           : {n_splits}")
print(f"Subject-overlap failures: {violations}")
print("PASS - no leakage." if violations == 0 else "FAIL - leakage detected!")

### 6.3 Classification

In [ ]:
TASKS = {
    "AD_vs_CN":        ["A", "C"],
    "AD_vs_FTD":       ["A", "F"],
    "AD_vs_FTD_vs_CN": ["A", "F", "C"],
}

results_B = {}
for task_name, classes in TASKS.items():
    print(f"\n{'=' * 70}\nTASK: {task_name}   classes={classes}\n{'=' * 70}")
    try:
        res = ds.run_cross_validation(features_B, feature_names, classes, cfg,
                                      classifier=cfg.primary_classifier,
                                      scheme="sgkf", pipeline_label="B")
        if "error" in res:
            print("SKIPPED:", res["error"])
            results_B[task_name] = res
            continue
        s = res["summary"]
        print(f"Subjects: {res['n_subjects']}  {res['subjects_per_class']}")
        print(f"  Balanced accuracy : {s['balanced_accuracy_mean']:.3f} (SD {s['balanced_accuracy_std']:.3f})")
        print(f"  Accuracy          : {s['accuracy_mean']:.3f}")
        print(f"  Macro F1          : {s['f1_macro_mean']:.3f}")
        if "roc_auc_mean" in s:
            print(f"  ROC-AUC           : {s['roc_auc_mean']:.3f}")
            print(f"  Sensitivity       : {s['sensitivity_mean']:.3f}")
            print(f"  Specificity       : {s['specificity_mean']:.3f}")
        results_B[task_name] = res
    except Exception as exc:
        print(f"FAILED: {exc!r}")
        results_B[task_name] = {"error": repr(exc)}

ds.save_results(cfg, "classification_results_B", results_B)

### 6.4 Fold-level and out-of-fold results

In [ ]:
fold_rows = []
for task_name, res in results_B.items():
    for fr in res.get("fold_results", []):
        fold_rows.append({"pipeline": "B", "task": task_name, **fr})
fold_results_B = pd.DataFrame(fold_rows)
if not fold_results_B.empty:
    fold_results_B = fold_results_B.drop(
        columns=[c for c in ["confusion_matrix", "confusion_matrix_labels"]
                 if c in fold_results_B.columns])
ds.save_results(cfg, "fold_results_B", fold_results_B)

oof_rows = []
for task_name, res in results_B.items():
    for r in res.get("oof_predictions", []):
        oof_rows.append({"pipeline": "B", "task": task_name, **r})
oof_B = pd.DataFrame(oof_rows)
ds.save_results(cfg, "oof_predictions_B", oof_B)

if not fold_results_B.empty:
    display(fold_results_B.groupby("task")[
        ["balanced_accuracy", "f1_macro", "accuracy"]].agg(["mean", "std"]).round(3))
print(f"\nOut-of-fold predictions: {oof_B.shape}")

### 6.5 Secondary validation (LOSO) and secondary classifier

In [ ]:
loso_B = {}
if cfg.run_loso:
    for task_name, classes in TASKS.items():
        try:
            res = ds.run_cross_validation(features_B, feature_names, classes, cfg,
                                          classifier=cfg.primary_classifier,
                                          scheme="loso", pipeline_label="B")
            if "error" in res:
                print(f"{task_name:<18} skipped: {res['error']}")
                continue
            p = res["pooled_subject_level"]
            print(f"{task_name:<18} bal-acc {p['balanced_accuracy']:.3f}  "
                  f"macro-F1 {p['f1_macro']:.3f}  (n={res['n_subjects']})")
            loso_B[task_name] = res
        except Exception as exc:
            print(f"{task_name:<18} FAILED: {exc!r}")
    ds.save_results(cfg, "loso_results_B", loso_B)

In [ ]:
results_B_rf = {}
for task_name, classes in TASKS.items():
    try:
        res = ds.run_cross_validation(features_B, feature_names, classes, cfg,
                                      classifier=cfg.secondary_classifier,
                                      scheme="sgkf", pipeline_label="B")
        if "error" in res:
            print(f"{task_name:<18} skipped: {res['error']}")
            continue
        s = res["summary"]
        print(f"{task_name:<18} bal-acc {s['balanced_accuracy_mean']:.3f} "
              f"(SD {s['balanced_accuracy_std']:.3f})")
        results_B_rf[task_name] = res
    except Exception as exc:
        print(f"{task_name:<18} FAILED: {exc!r}")

ds.save_results(cfg, "classification_results_B_rf", results_B_rf)

### 6.6 Confusion matrices and ROC curves

In [ ]:
from sklearn.metrics import roc_curve, auc

valid = [t for t, r in results_B.items() if "pooled_subject_level" in r]
if valid:
    fig, axes = plt.subplots(1, len(valid), figsize=(5 * len(valid), 4.2))
    axes = np.atleast_1d(axes)
    for ax, task_name in zip(axes, valid):
        p = results_B[task_name]["pooled_subject_level"]
        cm = np.array(p["confusion_matrix"]); labs = p["confusion_matrix_labels"]
        ax.imshow(cm, cmap="Oranges")
        ax.set_xticks(range(len(labs)), labs); ax.set_yticks(range(len(labs)), labs)
        ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        ax.set_title(f"{task_name}\nbal-acc = {p['balanced_accuracy']:.3f}")
        thr = cm.max() / 2 if cm.max() else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > thr else "black", fontweight="bold")
        ax.grid(False)
    plt.suptitle("Pipeline B — subject-level confusion matrices (counts = subjects)", y=1.03)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "confusion_matrices_B.png", dpi=150, bbox_inches="tight")
    plt.show()

binary = [t for t in ["AD_vs_CN", "AD_vs_FTD"] if t in results_B and "oof_predictions" in results_B[t]]
if binary:
    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    for task_name in binary:
        oof = pd.DataFrame(results_B[task_name]["oof_predictions"])
        pos = sorted(TASKS[task_name])[1]
        y_true = (oof["true_class"] == pos).astype(int)
        if y_true.nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, oof[f"proba_{pos}"])
        ax.plot(fpr, tpr, lw=2, label=f"{task_name} (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    ax.set_xlabel("False positive rate (1 − specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title("Pipeline B — subject-level ROC curves")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "roc_curves_B.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 7. Preliminary paired comparison

A first look, so that anything obviously wrong surfaces now rather than in Notebook 03.
The authoritative analysis — with effect sizes, confidence intervals and multiple
statistical approaches — lives in Notebook 03.

**Why a paired bootstrap over subjects rather than a t-test over folds?**
Cross-validation folds share training data, so fold-level metrics are positively
correlated and their variance is underestimated (Dietterich 1998; Nadeau & Bengio 2003).
Testing across folds inflates the false-positive rate. Subjects are the genuine
independent sampling units, and both pipelines scored the *same* subjects, which is what
makes the comparison properly paired.

In [ ]:
try:
    oof_A_all = pd.read_csv(cfg.results_dir / "oof_predictions_A.csv")
except FileNotFoundError:
    oof_A_all = None
    print("Pipeline A out-of-fold predictions not found. Run Notebook 01 first.")

if oof_A_all is not None:
    rows = []
    for task_name, classes in TASKS.items():
        a = oof_A_all[oof_A_all["task"] == task_name]
        b = oof_B[oof_B["task"] == task_name]
        if a.empty or b.empty:
            print(f"{task_name}: not computed (one pipeline has no results).")
            continue
        bs = ds.paired_bootstrap_subject_level(
            a, b, metric="balanced_accuracy", classes=sorted(classes),
            n_boot=2000, seed=cfg.random_seed)   # 2000 here; 10000 in Notebook 03
        if "error" in bs:
            print(f"{task_name}: {bs['error']}")
            continue
        rows.append({
            "task": task_name,
            "pipeline_A": round(bs["pipeline_A"], 4),
            "pipeline_B": round(bs["pipeline_B"], 4),
            "delta_B_minus_A": round(bs["observed_delta_B_minus_A"], 4),
            "ci95_low": round(bs["ci95_low"], 4),
            "ci95_high": round(bs["ci95_high"], 4),
            "p_value": round(bs["p_value_two_sided"], 4),
            "n_subjects": bs["n_subjects"],
        })

    prelim = pd.DataFrame(rows)
    if not prelim.empty:
        display(prelim)
        print("\nReading this table: if ci95_low <= 0 <= ci95_high, the data do not "
              "support a difference between the pipelines on that task. That is a "
              "legitimate and informative result, not a failed experiment.")
        for _, r in prelim.iterrows():
            verdict = ("no evidence of a difference"
                       if r["ci95_low"] <= 0 <= r["ci95_high"]
                       else ("B higher" if r["delta_B_minus_A"] > 0 else "A higher"))
            print(f"  {r['task']:<18} {verdict}")
    else:
        print("No paired comparisons could be computed.")

---
## 8. Computational benchmarking

In [ ]:
rows = []
for sid in subjects_ok_B:
    mp = cfg.cache_dir / "pipeline_B" / f"{sid}_meta.json"
    if not mp.exists():
        continue
    with open(mp) as fh:
        m = json.load(fh)
    if m.get("status") not in ("ok", "cached"):
        continue
    steps = m.get("preprocess_meta", {}).get("steps", {})
    rows.append({
        "participant_id": sid, "group": m.get("group"), "pipeline": "B",
        "input_duration_s": m.get("input_duration_s"),
        "load_s": m.get("timing_load", {}).get("wall_time_s"),
        "preprocess_s": m.get("timing_preprocess", {}).get("wall_time_s"),
        "epoch_s": m.get("timing_epoch", {}).get("wall_time_s"),
        "features_s": m.get("timing_features", {}).get("wall_time_s"),
        "total_s": m.get("timing_total", {}).get("wall_time_s"),
        "peak_rss_mb": m.get("timing_total", {}).get("peak_rss_mb"),
        "filter_s": steps.get("filter", {}).get("wall_time_s"),
        "badchan_s": steps.get("bad_channels", {}).get("wall_time_s"),
        "interp_s": steps.get("interpolate", {}).get("wall_time_s"),
        "n_epochs": m.get("n_epochs"),
    })

runtime_B = pd.DataFrame(rows)
if not runtime_B.empty:
    runtime_B["s_per_min_recording"] = (
        runtime_B["total_s"] / (runtime_B["input_duration_s"] / 60.0))
ds.save_results(cfg, "runtime_results_B", runtime_B)

if not runtime_B.empty:
    display(runtime_B[["total_s", "preprocess_s", "filter_s", "badchan_s", "interp_s",
                       "features_s", "peak_rss_mb"]].describe().round(2))
    print(f"\nMean total per subject : {runtime_B['total_s'].mean():.1f} s")
    print(f"Total for {len(runtime_B)} subjects : {runtime_B['total_s'].sum() / 60:.1f} min")

### 8.1 Head-to-head runtime

In [ ]:
try:
    runtime_A = pd.read_csv(cfg.results_dir / "runtime_results_A.csv")
except FileNotFoundError:
    runtime_A = None
    print("Pipeline A runtime file not found. Run Notebook 01 first.")

if runtime_A is not None and not runtime_B.empty:
    merged = runtime_A.merge(runtime_B, on="participant_id", suffixes=("_A", "_B"))
    if merged.empty:
        print("No subjects processed by both pipelines.")
    else:
        comp = pd.DataFrame({
            "metric": ["Preprocessing time (s)", "Total time (s)", "Peak RSS (MB)"],
            "pipeline_A": [merged["preprocess_s_A"].mean(), merged["total_s_A"].mean(),
                           merged["peak_rss_mb_A"].mean()],
            "pipeline_B": [merged["preprocess_s_B"].mean(), merged["total_s_B"].mean(),
                           merged["peak_rss_mb_B"].mean()],
        })
        comp["ratio_A_over_B"] = (comp["pipeline_A"] / comp["pipeline_B"]).round(2)
        comp[["pipeline_A", "pipeline_B"]] = comp[["pipeline_A", "pipeline_B"]].round(2)
        ds.save_results(cfg, "runtime_comparison_preliminary", comp)
        display(comp)
        print(f"\nPaired on {len(merged)} subjects processed by both pipelines, "
              f"in the same Colab session recorded in section 1.7.")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
        axes[0].scatter(merged["preprocess_s_A"], merged["preprocess_s_B"], alpha=0.75,
                        s=45, edgecolor="black")
        lim = max(merged["preprocess_s_A"].max(), merged["preprocess_s_B"].max()) * 1.08
        axes[0].plot([0, lim], [0, lim], "k--", lw=1, label="equal cost")
        axes[0].set_xlabel("Pipeline A preprocessing time (s)")
        axes[0].set_ylabel("Pipeline B preprocessing time (s)")
        axes[0].set_title("Per-subject preprocessing cost\n(points below the line: B is faster)")
        axes[0].legend()

        axes[1].boxplot([merged["preprocess_s_A"], merged["preprocess_s_B"]],
                        tick_labels=["Pipeline A", "Pipeline B"])
        axes[1].set_ylabel("Preprocessing time (s)")
        axes[1].set_title("Distribution of per-subject preprocessing time")
        plt.tight_layout()
        plt.savefig(cfg.figures_dir / "runtime_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()

### 8.2 Storage and scalability

In [ ]:
storage_B = {
    "cache_pipeline_B_mb": ds.directory_size_mb(cfg.cache_dir / "pipeline_B"),
    "n_subjects_cached": len(subjects_ok_B),
}
if subjects_ok_B:
    storage_B["mb_per_subject"] = round(
        storage_B["cache_pipeline_B_mb"] / len(subjects_ok_B), 3)
ds.save_results(cfg, "storage_B", storage_B)
for k, v in storage_B.items():
    print(f"{k:<28} {v}")

if not runtime_B.empty and len(runtime_B) >= 3:
    times = runtime_B["total_s"].to_numpy()
    n_max = len(times)
    checkpoints = [n for n in [5, 10, 20, 40, 88] if n <= n_max]
    if n_max not in checkpoints:
        checkpoints.append(n_max)
    cumulative = np.cumsum(times)
    rows = [{"n_subjects": n,
             "cumulative_runtime_s": round(cumulative[n - 1], 1),
             "cumulative_runtime_min": round(cumulative[n - 1] / 60, 2),
             "mean_runtime_per_subject_s": round(cumulative[n - 1] / n, 2),
             "peak_rss_mb": round(runtime_B["peak_rss_mb"].iloc[:n].max(), 1),
             "measured": True} for n in checkpoints]
    scal_B = pd.DataFrame(rows)
    coef = np.polyfit(scal_B["n_subjects"], scal_B["cumulative_runtime_s"], 1)
    est_88 = float(np.polyval(coef, 88))
    print(f"\nFitted slope       : {coef[0]:.2f} s per additional subject")
    print(f"Extrapolated to 88 : {est_88 / 60:.1f} min "
          f"({'MEASURED' if n_max >= 88 else 'ESTIMATE - beyond measured range'})")
    if n_max < 88:
        scal_B = pd.concat([scal_B, pd.DataFrame([{
            "n_subjects": 88, "cumulative_runtime_s": round(est_88, 1),
            "cumulative_runtime_min": round(est_88 / 60, 2),
            "mean_runtime_per_subject_s": round(est_88 / 88, 2),
            "peak_rss_mb": np.nan, "measured": False}])], ignore_index=True)
    ds.save_results(cfg, "scalability_B", scal_B)
    display(scal_B)
else:
    print("\nScalability: NOT COMPUTED (fewer than 3 subjects processed).")

---
## 9. Summary of Notebook 02

In [ ]:
summary_B = {
    "notebook": "02_alternative_pipeline_ds004504",
    "pipeline": "B (deterministic, ICA-free artifact handling)",
    "run_mode": cfg.mode(),
    "subjects_requested": len(subjects),
    "subjects_processed": len(subjects_ok_B),
    "subjects_failed": int((~ok_mask).sum()),
    "n_features": len(feature_names),
    "n_epochs_total": int(len(features_B)),
    "random_seed": cfg.random_seed,
}
for task_name, res in results_B.items():
    s = res.get("summary", {})
    summary_B[f"{task_name}_balanced_accuracy"] = (
        round(s["balanced_accuracy_mean"], 4) if "balanced_accuracy_mean" in s
        else "not computed")
if not runtime_B.empty:
    summary_B["mean_runtime_per_subject_s"] = round(runtime_B["total_s"].mean(), 2)
    summary_B["mean_peak_rss_mb"] = round(runtime_B["peak_rss_mb"].mean(), 1)

ds.save_results(cfg, "summary_B", summary_B)

print("=" * 70)
print("NOTEBOOK 02 SUMMARY - Pipeline B")
print("=" * 70)
for k, v in summary_B.items():
    print(f"{k:<38} {v}")
print("\nAll results written to:", cfg.results_dir)
print("\nNEXT: run 03_pipeline_comparison_ds004504.ipynb for the full statistical "
      "comparison and the final research conclusion.")

---
## 10. Notes and limitations of this notebook

**Pipeline B's own weaknesses, stated plainly:**

- **Interpolation on 19 electrodes is coarse.** Spherical-spline interpolation was designed for high-density montages; with 19 widely spaced electrodes, a reconstructed channel is a rough estimate. If many channels are flagged, Pipeline B may be *introducing* smoothing artifacts rather than removing noise.
- **No mechanism for ocular artifact.** Pipeline B has no component-based step, so eye-movement artifact is attenuated only insofar as the band-pass and epoch rejection catch it. The dataset README notes that eye-movement artifacts appear in some recordings *despite* the eyes-closed protocol. This is exactly the scenario in which de Cheveigné's rebuttal predicts ICA should help, so a finding that Pipeline A performs better here would be theoretically coherent, not anomalous.
- **The bad-channel thresholds (0.4 correlation, robust-z 5) are conventional but not tuned.** Tuning them against classification accuracy would be a form of leakage; leaving them at literature-standard defaults is the honest choice, at the cost of possible sub-optimality.

**Carried forward to Notebook 03:** `classification_results_B`, `fold_results_B`,
`oof_predictions_B`, `runtime_results_B`, `signal_quality_results_B`, `scalability_B`,
`bad_channel_stats_B`.